# Hansard RAG — retrieval evaluation


1. **Ground truth generation** — sample chunks, have Groq generate questions a user might ask that each chunk answers. Saved to `data/processed/ground_truth.csv` (one-off, re-used by every eval run after).
2. **Evaluation** — for each generated question, run each retrieval approach and check whether the source contribution comes back in the top k. Metrics: **hit rate** (is it in the top k?) and **MRR** (how high up?).

Approaches compared:
- keyword (BM25)
- vector (kNN)
- hybrid (RRF fusion)
- optional whole-contribution chunking

Needs `GROQ_API_KEY` in `.env`, Elasticsearch up, and the index built by `index.ipynb`.


In [ ]:
import json
import os
import time
import random
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from tqdm.auto import tqdm

from shared_funcs.search import keyword_search, vector_search, hybrid_search

load_dotenv()

CHUNKS_PATH = Path("data/processed/chunks.jsonl")
GROUND_TRUTH_PATH = Path("data/processed/ground_truth.csv")

GROQ_MODEL = "llama-3.3-70b-versatile"
N_SAMPLE_CHUNKS = 200      # chunks to generate questions from
QUESTIONS_PER_CHUNK = 5
MIN_CHUNK_WORDS = 50

client = Groq(api_key=os.environ["GROQ_API_KEY"])

## 1. Sample chunks

Random sample of substantial chunks. Seeded so the ground truth is reproducible.

In [ ]:
chunks = [json.loads(line) for line in CHUNKS_PATH.open()]
substantial = [c for c in chunks if len(c["text"].split()) >= MIN_CHUNK_WORDS]
print(f"{len(chunks)} chunks, {len(substantial)} substantial (>= {MIN_CHUNK_WORDS} words)")

random.seed(42)
sample = random.sample(substantial, min(N_SAMPLE_CHUNKS, len(substantial)))
len(sample)

## 2. Generate questions with Groq

Prompt asks for questions a member of the public might ask — phrased without copying the chunk's exact words.


In [ ]:
PROMPT_TEMPLATE = """You are helping evaluate a search system over UK parliamentary debates.

Below is an excerpt from a debate. Generate {n} questions that a member of the public
might ask, where this excerpt would be a good answer or highly relevant.

Rules:
- Questions must be answerable from the excerpt content
- Use natural everyday phrasing, NOT parliamentary language
- Do not copy distinctive phrases from the excerpt - paraphrase
- Vary the question styles (what/why/how/who)

Debate title: {title}
Speaker: {speaker}
Excerpt:
{text}

Respond with ONLY a JSON array of {n} strings, no other text."""


def generate_questions(chunk, n=QUESTIONS_PER_CHUNK, max_retries=5):
    prompt = PROMPT_TEMPLATE.format(
        n=n, title=chunk["debate_title"], speaker=chunk["speaker"], text=chunk["text"]
    )
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
            )
            raw = response.choices[0].message.content.strip()
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            questions = json.loads(raw)
            if isinstance(questions, list) and all(isinstance(q, str) for q in questions):
                return questions[:n]
        except json.JSONDecodeError:
            pass  # malformed output - retry
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                wait = 2 ** (attempt + 2)
                print(f"  rate limited, sleeping {wait}s")
                time.sleep(wait)
            else:
                raise
    return []  # give up on this chunk after retries

In [ ]:
# One-off generation: skips itself if the CSV already exists (delete it to regenerate)
if GROUND_TRUTH_PATH.exists():
    ground_truth = pd.read_csv(GROUND_TRUTH_PATH)
    print(f"loaded existing ground truth: {len(ground_truth)} questions")
else:
    rows = []
    for chunk in tqdm(sample):
        for q in generate_questions(chunk):
            rows.append(
                {
                    "question": q,
                    "chunk_id": chunk["chunk_id"],
                    "item_id": chunk["chunk_id"].rsplit("_", 1)[0],
                    "debate_ext_id": chunk["debate_ext_id"],
                }
            )
        time.sleep(1.2)  # stay under free-tier requests/minute
    ground_truth = pd.DataFrame(rows)
    ground_truth.to_csv(GROUND_TRUTH_PATH, index=False)
    print(f"generated {len(ground_truth)} questions -> {GROUND_TRUTH_PATH}")

ground_truth.sample(5)

## 3. Evaluation harness

A retrieved chunk counts as **relevant if it comes from the same contribution** (`item_id`) as the chunk the question was generated from — not the exact chunk. This matters for fairness: it stops the paragraph-vs-contribution comparison being biased by chunk granularity, since both strategies share the same contribution IDs.

- **Hit rate** = fraction of questions where a relevant chunk appears in the top k
- **MRR** = mean of 1/rank of the first relevant chunk (0 if absent) - rewards ranking it *high*, not just retrieving it

In [ ]:
def evaluate(search_fn, ground_truth_df, k=5):
    hits, reciprocal_ranks = [], []
    for _, row in tqdm(ground_truth_df.iterrows(), total=len(ground_truth_df)):
        results = search_fn(row["question"], k=k)
        result_items = [r["chunk_id"].rsplit("_", 1)[0] for r in results]
        try:
            rank = result_items.index(row["item_id"]) + 1
            hits.append(1)
            reciprocal_ranks.append(1 / rank)
        except ValueError:
            hits.append(0)
            reciprocal_ranks.append(0)
    return {
        "hit_rate": sum(hits) / len(hits),
        "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks),
    }

In [ ]:
# For contribution evaluation, uncomment

# import shared_funcs.search as S
# S.INDEX_NAME = "hansard-chunks-contribution"
# print("evaluating against:", S.INDEX_NAME)

## 4. Run the comparison


In [ ]:


results = {}
for name, fn in [("keyword", keyword_search), ("vector", vector_search), ("hybrid", hybrid_search)]:
    print(f"evaluating {name}...")
    results[name] = evaluate(fn, ground_truth, k=5)

summary = pd.DataFrame(results).T
summary

## 5. (Optional) chunking strategy comparison

Build a second index from the whole-contribution chunks, point the search functions at it, and evaluate against the *same* ground truth (fair because relevance is judged at contribution level):

1. `python`-side: re-run `transform` with `strategy="contribution"` -> `chunks_contribution.jsonl`
2. re-run `index.ipynb` with `CHUNKS_PATH` and `INDEX_NAME` switched (e.g. `hansard-chunks-contribution`)
3. re-run section 4 here with the search functions pointed at that index

Record both tables in the README - this is the experiment that answers whether splitting long speeches actually improves retrieval.

In [ ]:
# Persist results for the README / later comparison

# For contribution evaluation, swap commented lines
summary.to_csv("data/processed/retrieval_eval_paragraph.csv")
# summary.to_csv("data/processed/retrieval_eval_contribution.csv")
summary